# AgriShield

End-to-end soil-risk pipeline: harmonise surveys, train a classifier, query live satellite + climate for a new site.

```
LUCAS + WoSIS  ->  data/agrishield_training.csv  ->  RandomForest
New lat/lon    ->  GEE (Sentinel-2, WorldClim, static soil)  ->  risk report
```

**Hard rule this notebook follows:** every column in `FEATURE_COLUMNS` (agrishield/config.py) must be obtainable live, from lat/lon alone, with no lab test. Lab-only chemistry (OC, N, P, K, EC, CEC, bulk density) is the TARGET, never a feature. `dataset.py` and `model.py` are unchanged from before; `config.py`, `gee.py`, `climate.py`, `inference.py` were rewritten to enforce this.

Run cells top to bottom. Section 2b (satellite/climate enrichment) is the slow one -- smoke test before the full run.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agrishield.config import TRAINING_CSV, FEATURE_COLUMNS, EXAMPLE_SITE
from agrishield.dataset import build_training_csv
from agrishield.climate import enrich_training_csv
from agrishield.model import train, save_model, predict_proba
from agrishield.inference import live_features, live_model_inputs

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
print("root:", ROOT)
print("training table exists:", TRAINING_CSV.exists())

root: d:\Jupyter Lab\Agrishield AI
training table exists: True


## 2. Base training table -- LUCAS + WoSIS only

No APIs yet, pure local files. Fast (seconds, not minutes).

`sample_year` is kept as a lookup key for step 2b -- it decides which satellite image to fetch for each row. It is never a model feature; check `FEATURE_COLUMNS` above if you want to confirm.

In [2]:
df = build_training_csv()
print("rows:", len(df), " cols:", df.shape[1])
print(df["source"].value_counts())
display(df.head())

rows: 250579  cols: 38
source
wosis         195012
lucas_2009     19791
lucas_2018     18983
lucas_2015     16793
Name: count, dtype: int64


,source,sample_id,latitude,longitude,country,continent,sample_year,ph_h2o,ph_cacl2,oc_gkg,oc_20_30_gkg,n_gkg,p_mgkg,k_mgkg,ec,caco3,caco3_20_30,ox_al,ox_fe,clay_pct,sand_pct,silt_pct,coarse_pct,elevation_m,bd_0_20,land_cover,land_cover_l1,land_use,depth,nuts1,nuts2,nuts3,soil_stones,cec_ph7,totc_gkg,acidic,low_oc,soil_stress
0,lucas_2018,47862690,47.150238,16.134212,AT,Europe,2018.0,4.81,4.1,12.4,NaN,1.1,NaN,101.9,8.73,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,291.0,NaN,Woodland,Other coniferous woodland,Forestry,0-20 cm,AT1,AT11,AT113,NaN,NaN,NaN,1.0,0.0,1.0
1,lucas_2018,47882704,47.274272,16.175359,AT,Europe,2018.0,4.93,4.1,16.7,NaN,1.3,NaN,51.2,5.06,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,373.0,NaN,Woodland,Spruce dominated coniferous woodland,Forestry,0-20 cm,AT1,AT11,AT113,NaN,NaN,NaN,1.0,0.0,1.0
2,lucas_2018,47982688,47.123260,16.289693,AT,Europe,2018.0,4.85,4.1,47.5,NaN,3.1,12.3,114.8,12.53,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,246.0,NaN,Woodland,Other mixed woodland,Forestry,0-20 cm,AT1,AT11,AT113,NaN,NaN,NaN,1.0,0.0,1.0
3,lucas_2018,48022702,47.245693,16.357506,AT,Europe,2018.0,5.80,5.5,28.1,NaN,2.0,NaN,165.8,21.10,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,305.0,NaN,Woodland,Pine dominated coniferous woodland,Forestry,0-20 cm,AT1,AT11,AT113,NaN,NaN,NaN,0.0,0.0,0.0
4,lucas_2018,48062708,47.296372,16.416782,AT,Europe,2018.0,6.48,6.1,19.4,NaN,2.2,NaN,42.1,10.89,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,335.0,NaN,Woodland,Pine dominated coniferous woodland,Forestry,0-20 cm,AT1,AT11,AT113,NaN,NaN,NaN,0.0,0.0,0.0


### 2a. Know your date coverage before spending GEE quota

Sentinel-2 (the satellite you're querying in 2b) only exists from 2015 onward. Rows with an older or missing `sample_year` will end up with satellite columns as `NaN` -- expected, not a bug. WorldClim and static soil/elevation don't depend on date, so they'll still fill in for almost every row.

In [3]:
has_year = df["sample_year"].notna()
print("missing sample_year:", df["sample_year"].isna().sum(),
      f"({df['sample_year'].isna().mean()*100:.1f}%)")
print("year < 2015 (pre-Sentinel-2):", (df.loc[has_year, "sample_year"] < 2015).sum())
print("year >= 2015 (real spectral data possible):", (df.loc[has_year, "sample_year"] >= 2015).sum())

missing sample_year: 94428 (37.7%)
year < 2015 (pre-Sentinel-2): 120226
year >= 2015 (real spectral data possible): 35925


## 2b. Attach satellite + climate + soil columns (slow -- do the smoke test first)

Three things get attached to the SAME rows as new columns:
- **Sentinel-2 bands + NDVI** -- batched per `sample_year` (one composite image per year, not one API call per row). Only fills for rows with `sample_year >= 2015`.
- **Static soil texture + elevation** -- from OpenLandMap/SRTM. Fills for virtually every row with coordinates, and OVERWRITES the LUCAS lab-measured `clay_pct`/`sand_pct`/`silt_pct`/`elevation_m` so training and live inference read from the identical source.
- **WorldClim** -- climatology, fills for virtually every row.

Run the 800-row smoke test first. Check the fill rates make sense. Only then uncomment the full run.

In [ ]:
# # SMOKE TEST -- ~all rows, should take a couple of minutes to hours
# _smoke = enrich_training_csv(batch_size=400, max_rows=None)
# check_cols = ["B2", "B3", "B4", "ndvi", "clay_pct", "sand_pct", "elevation_m", "tmean_c", "precip_mm"]
# print(_smoke[check_cols].notna().mean().round(3))

  rows 0-400 / 250579
  rows 400-800 / 250579
  rows 800-1200 / 250579
  rows 1200-1600 / 250579
  rows 1600-2000 / 250579
  rows 2000-2400 / 250579
  rows 2400-2800 / 250579
  rows 2800-3200 / 250579
  rows 3200-3600 / 250579
  rows 3600-4000 / 250579
  rows 4000-4400 / 250579
  rows 4400-4800 / 250579
  rows 4800-5200 / 250579
  rows 5200-5600 / 250579
  rows 5600-6000 / 250579
  rows 6000-6400 / 250579
  rows 6400-6800 / 250579
  rows 6800-7200 / 250579
  rows 7200-7600 / 250579
  rows 7600-8000 / 250579
  rows 8000-8400 / 250579
  rows 8400-8800 / 250579
  rows 8800-9200 / 250579
  rows 9200-9600 / 250579
  rows 9600-10000 / 250579
  rows 10000-10400 / 250579
  rows 10400-10800 / 250579
  rows 10800-11200 / 250579
  rows 11200-11600 / 250579
  rows 11600-12000 / 250579
  rows 12000-12400 / 250579
  rows 12400-12800 / 250579
  rows 12800-13200 / 250579
  rows 13200-13600 / 250579
  rows 13600-14000 / 250579
  rows 14000-14400 / 250579
  rows 14400-14800 / 250579
  rows 14800-15200 /

## 3. Train

Loads from disk (not the in-memory `df` above) so this cell works even if you restarted the kernel after the overnight run finished.

In [5]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("training on", len(df), "rows")
model, report = train(df)
print(report)
save_model(model)

import numpy as np
cols = [c for c in FEATURE_COLUMNS if c in df.columns]
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=cols).sort_values(ascending=False)
print("\nfeature importances:")
print(importances)

training on 250578 rows
class balance (target = acidic):
acidic
0.0    0.694
1.0    0.306
              precision    recall  f1-score   support

         0.0      0.864     0.920     0.891     26652
         1.0      0.785     0.667     0.722     11604

    accuracy                          0.844     38256
   macro avg      0.825     0.794     0.806     38256
weighted avg      0.840     0.844     0.840     38256


feature importances:
precip_mm             0.205702
temp_seasonality      0.119735
tmean_c               0.113232
clay_pct              0.104580
elevation_m           0.094979
slope_deg             0.091237
precip_seasonality    0.085452
sand_pct              0.073362
silt_pct              0.059604
B12                   0.008802
B3                    0.006919
B11                   0.006845
B4                    0.005222
B2                    0.004963
ndvi                  0.004779
B5                    0.004678
B6                    0.003359
B7                    0.003325
B8 

### 3b. Optional comparison: satellite-era rows only (sample_year >= 2015)

You chose to train on everything first and check accuracy -- good, evidence beats guessing. This cell runs the same model on just the ~36k rows that have REAL (not imputed) spectral bands, so you can see whether accuracy actually improves on cleaner data or whether the extra 196k rows were pulling their weight.

In [6]:
df_recent = df[df["sample_year"] >= 2015].copy()
print("rows with real satellite era data:", len(df_recent))
model_recent, report_recent = train(df_recent)
print("--- full 232k ---")
print(report)
print("--- satellite-era only (~36k) ---")
print(report_recent)

rows with real satellite era data: 35925
class balance (target = acidic):
acidic
0.0    0.652
1.0    0.348
--- full 232k ---
              precision    recall  f1-score   support

         0.0      0.864     0.920     0.891     26652
         1.0      0.785     0.667     0.722     11604

    accuracy                          0.844     38256
   macro avg      0.825     0.794     0.806     38256
weighted avg      0.840     0.844     0.840     38256

--- satellite-era only (~36k) ---
              precision    recall  f1-score   support

         0.0      0.851     0.932     0.890      4572
         1.0      0.857     0.712     0.778      2603

    accuracy                          0.853      7175
   macro avg      0.854     0.822     0.834      7175
weighted avg      0.853     0.853     0.849      7175



## 4. Live inference for a new site

This is what actually runs when a farmer taps Calculate -- one coordinate, live GEE + weather calls, no training data touched. `live_model_inputs` feeds the model; `live_features` wraps that plus current weather for a human-facing report.

In [7]:
site = EXAMPLE_SITE
report_data = live_features(site["latitude"], site["longitude"])
print(site["name"], site["latitude"], site["longitude"])
report_data

Clayton, Victoria -37.91 145.13


{'latitude': -37.91,
 'longitude': 145.13,
 'model_inputs': {'B2': 296,
  'B3': 453.5,
  'B4': 452.5,
  'B5': 1122.5,
  'B6': 2031,
  'B7': 2326,
  'B8': 2344.5,
  'B11': 1574,
  'B12': 1179,
  'ndvi': 0.6764390468597412,
  'elevation_m': 101,
  'slope_deg': 0.3743423819541931,
  'clay_pct': 16,
  'sand_pct': 70,
  'silt_pct': 14.0,
  'tmean_c': 14.4,
  'temp_seasonality': 3660,
  'precip_mm': 836,
  'precip_seasonality': 17},
 'weather_now': {'temp_c': 16.42,
  'humidity': 66,
  'pressure': 1023,
  'wind_speed': 3.15,
  'rain_1h_mm': 0,
  'description': 'broken clouds'}}

In [8]:
inputs = live_model_inputs(site["latitude"], site["longitude"])
print(inputs)
predict_proba(model, inputs)

{'B2': 296, 'B3': 453.5, 'B4': 452.5, 'B5': 1122.5, 'B6': 2031, 'B7': 2326, 'B8': 2344.5, 'B11': 1574, 'B12': 1179, 'ndvi': 0.6764390468597412, 'elevation_m': 101, 'slope_deg': 0.3743423819541931, 'clay_pct': 16, 'sand_pct': 70, 'silt_pct': 14.0, 'tmean_c': 14.4, 'temp_seasonality': 3660, 'precip_mm': 836, 'precip_seasonality': 17}


{'predicted': 0, 'probability': {0: 0.58, 1: 0.42}}

In [9]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
check_cols = ["B2","B3","B4","ndvi","clay_pct","sand_pct","silt_pct","elevation_m","tmean_c","precip_mm"]
print(df[check_cols].notna().mean().round(3))
print(df.groupby(df["sample_year"] >= 2015)[check_cols].apply(lambda x: x.notna().mean()))

B2             0.076
B3             0.076
B4             0.076
ndvi           0.076
clay_pct       0.993
sand_pct       0.993
silt_pct       0.997
elevation_m    0.960
tmean_c        0.998
precip_mm      0.998
dtype: float64
                   B2        B3        B4      ndvi  clay_pct  sand_pct  \
sample_year                                                               
False        0.000000  0.000000  0.000000  0.000000  0.992667  0.992667   
True         0.531413  0.531413  0.531413  0.531413  0.997857  0.997857   

             silt_pct  elevation_m   tmean_c  precip_mm  
sample_year                                              
False        0.997373     0.972220  0.997340   0.997340  
True         0.997996     0.885177  0.998831   0.998831  


In [10]:
import pandas as pd
from agrishield.config import TRAINING_CSV
from agrishield.model import train, save_model, available_features

df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("training on", len(df), "rows")

model, report = train(df)
print(report)
save_model(model)

cols = available_features(df)
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=cols).sort_values(ascending=False)
print("\nfeature importances:")
print(importances)

training on 250578 rows
class balance (target = acidic):
acidic
0.0    0.694
1.0    0.306
              precision    recall  f1-score   support

         0.0      0.864     0.920     0.891     26652
         1.0      0.785     0.667     0.722     11604

    accuracy                          0.844     38256
   macro avg      0.825     0.794     0.806     38256
weighted avg      0.840     0.844     0.840     38256


feature importances:
precip_mm             0.205702
temp_seasonality      0.119735
tmean_c               0.113232
clay_pct              0.104580
elevation_m           0.094979
slope_deg             0.091237
precip_seasonality    0.085452
sand_pct              0.073362
silt_pct              0.059604
B12                   0.008802
B3                    0.006919
B11                   0.006845
B4                    0.005222
B2                    0.004963
ndvi                  0.004779
B5                    0.004678
B6                    0.003359
B7                    0.003325
B8 